# Day 1, Session 4 -- Demos: Lab Review & Integration

This capstone session integrates all Day 1 concepts -- LLM foundations (Session 1), prompt engineering (Session 2), and model evaluation (Session 3) -- into practical consulting AI workflows.

| Demo | Pattern | Why It Matters |
|------|---------|----------------|
| 1 | End-to-End Consulting Pipeline | Combines API calls, structured prompting, and G-Eval scoring into one workflow |
| 2 | Cost-Quality Trade-off Analysis | Balances quality against cost and latency for production systems |
| 3 | Day 2 Preview -- Raw API vs LangChain | Previews the abstraction power of chains for reusable pipelines |

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import openai
import json
import os
import time
import numpy as np
import pandas as pd

client = openai.OpenAI()
print(f"Model: {os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')}")
print("Setup complete.")

---
## Demo 1: End-to-End Consulting Analysis Pipeline

This demo combines API calls (Session 1), structured prompting (Session 2), and automated evaluation (Session 3) into a single pipeline that generates and quality-checks a consulting deliverable.

**What to observe:**
- How `g_eval_quick` reuses Session 3's G-Eval pattern for automated scoring
- How the pipeline chains generation -> evaluation -> verdict
- The bar-chart visualization of quality scores

In [ ]:
# Demo 1: End-to-End Consulting Analysis Pipeline
# Combines: API calls (S1) + Structured prompting (S2) + G-Eval scoring (S3)

def g_eval_quick(question, response, criterion, description):
    """Lightweight G-Eval scorer (from Session 3)."""
    prompt = f"""Rate this consulting response on {criterion} (1-5).
Criterion: {description}
Question: {question}
Response: {response}
Return JSON with 'score' (1-5) and 'reasoning' (one sentence)."""
    result = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        temperature=0, max_tokens=200
    )
    return json.loads(result.choices[0].message.content)


def consulting_pipeline(question, persona='McKinsey senior consultant'):
    """Generate a consulting analysis, then auto-evaluate it."""
    print(f"Question: {question}")
    print(f"Persona: {persona}")
    print("-" * 50)
    
    # Step 1: Generate (Session 1 + Session 2 techniques)
    start = time.time()
    response = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[
            {'role': 'system', 'content': f'You are a {persona}. Provide structured, MECE, data-driven analysis with specific recommendations.'},
            {'role': 'user', 'content': question}
        ],
        temperature=0, max_tokens=500
    )
    latency = round((time.time() - start) * 1000)
    content = response.choices[0].message.content
    tokens = response.usage.total_tokens
    
    print(f"\nGenerated Response ({latency}ms, {tokens} tokens):")
    print(content[:300] + '...' if len(content) > 300 else content)
    
    # Step 2: Evaluate (Session 3 techniques)
    criteria = {
        'MECE Structure': 'Response breaks down the problem into mutually exclusive, collectively exhaustive categories.',
        'Actionability': 'Recommendations are specific, prioritized, and implementable.',
        'Executive Readiness': 'Output is polished enough for a C-suite presentation.'
    }
    
    print(f"\nQuality Evaluation:")
    total = 0
    for crit_name, crit_desc in criteria.items():
        result = g_eval_quick(question, content, crit_name, crit_desc)
        score = result.get('score', 0)
        total += score
        bar = chr(9608) * score + chr(9617) * (5 - score)
        print(f"  {crit_name:20s} {bar} {score}/5")
    
    avg = round(total / len(criteria), 1)
    verdict = 'PARTNER-READY' if avg >= 4.0 else 'NEEDS REVISION' if avg >= 3.0 else 'REWORK'
    print(f"\n  Overall: {avg}/5.0 | Verdict: {verdict}")
    return {'content': content, 'avg_score': avg, 'verdict': verdict, 'latency_ms': latency, 'tokens': tokens}


# Run the pipeline
result = consulting_pipeline(
    'What are the key value creation levers for a PE-backed healthcare services platform?'
)

**Observe:** The pipeline generates a structured consulting response, then auto-evaluates it on three McKinsey-relevant criteria. The verdict (PARTNER-READY / NEEDS REVISION / REWORK) mimics a real quality gate.

---
## Demo 2: Cost-Quality Trade-off Analysis

For production consulting AI, we need to balance quality against cost and latency. This demo runs the same query at different configurations and compares the trade-offs.

**What to observe:**
- How temperature and max_tokens affect output quality, latency, and cost
- The cost difference between configurations is small with gpt-4o-mini but scales at volume

In [ ]:
# Demo 2: Cost-Quality Trade-off Analysis
# Combines: Cost estimation (S1) + Temperature comparison (S1) + Evaluation (S3)

PRICING = {
    'gpt-4o-mini': {'input': 0.15 / 1_000_000, 'output': 0.60 / 1_000_000},
    'gpt-4o': {'input': 2.50 / 1_000_000, 'output': 10.00 / 1_000_000},
}

question = 'How should a mid-size European bank approach digital transformation to improve customer acquisition?'
configs = [
    {'temperature': 0.0, 'max_tokens': 200, 'label': 'Fast/Cheap (T=0, 200 tok)'},
    {'temperature': 0.0, 'max_tokens': 500, 'label': 'Standard (T=0, 500 tok)'},
    {'temperature': 0.7, 'max_tokens': 500, 'label': 'Creative (T=0.7, 500 tok)'},
]

results = []
for config in configs:
    start = time.time()
    resp = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[
            {'role': 'system', 'content': 'You are a McKinsey digital strategy expert.'},
            {'role': 'user', 'content': question}
        ],
        temperature=config['temperature'],
        max_tokens=config['max_tokens']
    )
    latency = round((time.time() - start) * 1000)
    content = resp.choices[0].message.content
    usage = resp.usage
    
    score = g_eval_quick(question, content, 'Overall Quality',
        'Response is structured, actionable, and ready for executive review.')['score']
    
    model_name = os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')
    cost = 0
    if model_name in PRICING:
        cost = usage.prompt_tokens * PRICING[model_name]['input'] + usage.completion_tokens * PRICING[model_name]['output']
    
    results.append({
        'Config': config['label'], 'Quality': f"{score}/5",
        'Latency': f"{latency}ms", 'Tokens': usage.total_tokens,
        'Cost': f"${cost:.6f}", 'Response Length': len(content),
    })

df = pd.DataFrame(results)
print('COST-QUALITY TRADE-OFF ANALYSIS')
print('=' * 70)
print(df.to_string(index=False))
print('\nObservation: Higher max_tokens increases quality but also cost.')
print('Temperature 0.7 may add creativity but reduces consistency.')

**Observe:** At scale (thousands of queries per engagement), even small cost differences compound. The Standard config usually offers the best quality-to-cost ratio.

---
## Demo 3: Day 2 Preview -- From Raw API Calls to LangChain

Today we used raw OpenAI API calls. On Day 2, we will use **LangChain** to compose reusable pipelines. This demo shows the same task done both ways, previewing the abstraction power of chains.

**What to observe:**
- How the chain is parameterized with variables (`{practice}`, `{question}`)
- The same chain can be reused across different practices without rewriting

In [ ]:
# Demo 3: Day 2 Preview -- Raw API vs LangChain Pattern

# --- Approach A: Raw API (what we did today) ---
print('APPROACH A: Raw OpenAI API (Day 1 style)')
print('=' * 50)

system_msg = 'You are a McKinsey strategy consultant. Be concise and structured.'
user_msg = 'List 3 key risks in a cross-border pharmaceutical acquisition.'

response = client.chat.completions.create(
    model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
    messages=[
        {'role': 'system', 'content': system_msg},
        {'role': 'user', 'content': user_msg}
    ],
    temperature=0, max_tokens=300
)
print(response.choices[0].message.content)

# --- Approach B: LangChain pattern (Day 2 preview) ---
print(f'\n{"=" * 50}')
print('APPROACH B: LangChain LCEL Chain (Day 2 style)')
print('=' * 50)

try:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are a McKinsey {practice} consultant. Be concise and structured.'),
        ('user', '{question}')
    ])

    llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'), temperature=0)
    chain = prompt | llm | StrOutputParser()

    result = chain.invoke({'practice': 'strategy', 'question': user_msg})
    print(result)
    
    print('\nKey advantage: The chain is REUSABLE with different variables.')
    result2 = chain.invoke({'practice': 'operations', 'question': 'What are 3 supply chain optimization levers for a CPG company?'})
    print(f'\nOperations example: {result2[:150]}...')

except ImportError:
    print('LangChain not installed. Run: pip install langchain langchain-openai')
    print('You will set this up fully on Day 2.')

print('\n' + '=' * 50)
print('Day 2 will cover: LCEL chains, custom tools, StateGraph workflows, multi-agent systems')

**Observe:** The LangChain chain uses the pipe operator (`|`) to compose prompt, LLM, and parser into a reusable pipeline. Changing `{practice}` gives a different persona without rewriting any code. Day 2 will make this pattern central to everything we build.

---
## Summary

| Demo | Pattern | Key Takeaway |
|------|---------|-------------|
| 1 | End-to-End Pipeline | Combine generation + evaluation for auto quality gates |
| 2 | Cost-Quality Trade-off | Model configuration affects quality, latency, and cost at scale |
| 3 | Raw API vs LangChain | Chains make pipelines reusable and composable (Day 2 preview) |

**Key functions to reuse in exercises:**
- `g_eval_quick()` -- lightweight G-Eval scorer for any criterion
- `consulting_pipeline()` -- full generate-evaluate pipeline
- `PRICING` dict -- cost estimation per model